# Housekeeping

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving fake_job_postings.csv to fake_job_postings.csv


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Input, Dense, Lambda
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split

from transformers import BertTokenizer, TFBertModel
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset, random_split
from torch import nn, optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score



In [ ]:
df = pd.read_csv('fake_job_postings.csv')
df.head(5)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


# Data Cleaning

In [ ]:
df.columns

Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='object')

In [ ]:
# Fill NA with empty space
df.fillna(' ', inplace = True)

# Concatenate all text features into one big text
df['text'] = df['title'] + ". " + df['company_profile'] + ". " + df['description'] + ". " + df['requirements'] + ". " + df['benefits']

# Dropp all but text and fraudulent columns
df = df.drop(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function'], axis = 1)

In [ ]:
df.iloc[0]['text']

"Marketing Intern. We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly curated information about food and cooking. We attract the most talented home cooks and contributors in the country; we also publish well-known professionals like Mario Batali, Gwyneth Paltrow, and Danny Meyer. And we have partnerships with Whole Foods Market and Random House.Food52 has been named the best food website by the James Beard Foundation and IACP, and has been featured in the New York Times, NPR, Pando Daily, TechCrunch, and on the Today Show.We're located in Chelsea, in New York City.. Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and 

# BERT Modeling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size = 0.20, random_state = 1)


In [ ]:
X_train.tolist()

['Business Developer Austria - Switzerland. Massive Media\xa0is the social media company behind the successful digital brands\xa0#URL_18234f381f5e7b9a9ffdc727cd05c9046edffb45bce85533c8f9b6d0216e925e#\xa0and\xa0#URL_af2b2f34d003dd6238fb60ec002a2f9df551ec9f8c6df8c980fc4fd8d24cc707#. In November 2013 Massive Media bought and relaunched the social discovery platform\xa0Stepout. We enable members to meet nearby people instantly. Over 100 million people have joined our sites on web and mobile.. FunctionWe’re more than a normal website – we’re a social community platform with a unified mission to create unexpected ways of online advertising that change brand perception. We’re growing rapidly and have a variety of European and national accounts.  We’re looking for someone with experience in selling of online media campaigns for multiple clients that goes beyond the banner. Strong interest and an in-depth understanding of the (digital) media landscape, including emerging media and social networ

In [ ]:
checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [ ]:
x_train_tokenized = bert_tokenizer(X_train.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')


In [ ]:

x_test_tokenized = bert_tokenizer(X_test.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')


In [ ]:
MAX_SEQUENCE_LENGTH = 128                 # set max_length of the input sequence

def create_bert_classification_model(bert_model,
                                     num_train_layers=0,
                                     hidden_size = 200,
                                     dropout=0.3,
                                     learning_rate=0.00005):
    """
    Build a simple classification model with BERT. Use the Pooler Output for classification purposes
    """
    if num_train_layers == 0:
        # Freeze all layers of pre-trained BERT model
        bert_model.trainable = False

    elif num_train_layers == 12:
        # Train all layers of the BERT model
        bert_model.trainable = True

    else:
        # Restrict training to the num_train_layers outer transformer layers
        retrain_layers = []

        for retrain_layer_number in range(num_train_layers):

            layer_code = '_' + str(11 - retrain_layer_number)
            retrain_layers.append(layer_code)


        print('retrain layers: ', retrain_layers)

        for w in bert_model.weights:
            if not any([x in w.name for x in retrain_layers]):
                #print('freezing: ', w)
                w._trainable = False

    input_ids = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='input_ids_layer')
    token_type_ids = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='token_type_ids_layer')
    attention_mask = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='attention_mask_layer')

    bert_inputs = {'input_ids': input_ids,
                   'token_type_ids': token_type_ids,
                   'attention_mask': attention_mask}

    bert_out = bert_model(bert_inputs)

    pooler_token = bert_out[1]
    #cls_token = bert_out[0][:, 0, :]

    hidden = tf.keras.layers.Dense(hidden_size, activation='relu', name='hidden_layer')(pooler_token)


    hidden = tf.keras.layers.Dropout(dropout)(hidden)


    classification = tf.keras.layers.Dense(1, activation='sigmoid',name='classification_layer')(hidden)

    classification_model = tf.keras.Model(inputs=[input_ids, token_type_ids, attention_mask], outputs=[classification])

    classification_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                                 loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
                                 metrics='accuracy')

    return classification_model

In [ ]:
#let's get a fresh instance of the bert_model -- good practice
bert_model = TFBertModel.from_pretrained(checkpoint)
bert_classification_model = create_bert_classification_model(bert_model, num_train_layers=0)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [ ]:
bert_classification_model_history = bert_classification_model.fit(
    [x_train_tokenized.input_ids, x_train_tokenized.token_type_ids, x_train_tokenized.attention_mask],
    y_train,
    validation_data=([x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask], y_test),
    batch_size=32,
    epochs=2
)

Epoch 1/2
447/447 [==============================] - 190s 405ms/step - loss: 0.2203 - accuracy: 0.9414 - val_loss: 0.1727 - val_accuracy: 0.9572
Epoch 2/2
447/447 [==============================] - 176s 394ms/step - loss: 0.1995 - accuracy: 0.9502 - val_loss: 0.1770 - val_accuracy: 0.9572


In [ ]:
len(X_test), sum(y_test)

(3576, 153)

In [ ]:
153/3576

0.04278523489932886

In [ ]:
from sklearn.metrics import classification_report

# Assuming you have already trained your model and have predictions
y_pred = bert_classification_model.predict([x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask])
y_pred_classes = (y_pred > 0.5).astype(int)  # Assuming you have a binary classification task and using a threshold of 0.5

# Print classification report
print(classification_report(y_test, y_pred_classes))

112/112 [==============================] - 34s 306ms/step
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      3423
           1       0.00      0.00      0.00       153

    accuracy                           0.96      3576
   macro avg       0.48      0.50      0.49      3576
weighted avg       0.92      0.96      0.94      3576



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


BOW

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import classification_report

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to BoW vectors
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Choose a classifier (e.g., Naive Bayes)
classifier = MultinomialNB()

# Train the classifier
classifier.fit(X_train_bow, y_train)

# Predictions
y_pred = classifier.predict(X_test_bow)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       0.86      0.56      0.68       181

    accuracy                           0.97      3576
   macro avg       0.92      0.78      0.83      3576
weighted avg       0.97      0.97      0.97      3576



In [ ]:
from sklearn.svm import SVC

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to BoW vectors
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Choose a Support Vector Machine (SVM) classifier
classifier = SVC(kernel='linear', C=1.0)  # You can experiment with different kernel functions and parameters

# Train the classifier
classifier.fit(X_train_bow, y_train)

# Predictions
y_pred = classifier.predict(X_test_bow)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3395
           1       0.83      0.78      0.80       181

    accuracy                           0.98      3576
   macro avg       0.91      0.89      0.90      3576
weighted avg       0.98      0.98      0.98      3576



In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to BoW vectors
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Choose a Random Forest classifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42)  # You can experiment with the number of trees and other parameters

# Train the classifier
classifier.fit(X_train_bow, y_train)

# Predictions
y_pred = classifier.predict(X_test_bow)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       0.99      0.62      0.77       181

    accuracy                           0.98      3576
   macro avg       0.99      0.81      0.88      3576
weighted avg       0.98      0.98      0.98      3576



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to BoW vectors
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Choose a Gradient Boosting classifier
classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)  # You can experiment with the number of estimators, learning rate, and other parameters

# Train the classifier
classifier.fit(X_train_bow, y_train)

# Predictions
y_pred = classifier.predict(X_test_bow)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       1.00      0.62      0.77       181

    accuracy                           0.98      3576
   macro avg       0.99      0.81      0.88      3576
weighted avg       0.98      0.98      0.98      3576



In [ ]:
from sklearn.linear_model import PassiveAggressiveClassifier

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to BoW vectors
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Choose a Passive Aggressive Classifier
classifier = PassiveAggressiveClassifier(C=1.0, random_state=42)  # You can experiment with the regularization parameter (C) and other parameters

# Train the classifier
classifier.fit(X_train_bow, y_train)

# Predictions
y_pred = classifier.predict(X_test_bow)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3395
           1       0.81      0.75      0.78       181

    accuracy                           0.98      3576
   macro avg       0.90      0.87      0.88      3576
weighted avg       0.98      0.98      0.98      3576



TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import classification_report

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Choose a classifier (e.g., Naive Bayes)
classifier = MultinomialNB()

# Train the classifier
classifier.fit(X_train_tfidf, y_train)

# Predictions
y_pred = classifier.predict(X_test_tfidf)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3395
           1       0.00      0.00      0.00       181

    accuracy                           0.95      3576
   macro avg       0.47      0.50      0.49      3576
weighted avg       0.90      0.95      0.92      3576



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import classification_report

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Choose a Support Vector Machine (SVM) classifier
classifier = SVC(kernel='linear', C=1.0)  # You can experiment with different kernel functions and parameters

# Train the classifier
classifier.fit(X_train_tfidf, y_train)

# Predictions
y_pred = classifier.predict(X_test_tfidf)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       1.00      0.69      0.81       181

    accuracy                           0.98      3576
   macro avg       0.99      0.84      0.90      3576
weighted avg       0.98      0.98      0.98      3576



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import classification_report

# Assuming 'X' is a list of documents and 'y' is a list of corresponding labels
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Choose a Random Forest classifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42)  # You can experiment with the number of trees and other parameters

# Train the classifier
classifier.fit(X_train_tfidf, y_train)

# Predictions
y_pred = classifier.predict(X_test_tfidf)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       1.00      0.61      0.76       181

    accuracy                           0.98      3576
   macro avg       0.99      0.80      0.87      3576
weighted avg       0.98      0.98      0.98      3576



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Choose a Gradient Boosting classifier
classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Train the classifier
classifier.fit(X_train_tfidf, y_train)

# Predictions
y_pred = classifier.predict(X_test_tfidf)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       0.95      0.61      0.74       181

    accuracy                           0.98      3576
   macro avg       0.96      0.81      0.87      3576
weighted avg       0.98      0.98      0.98      3576



In [ ]:
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Choose a Passive Aggressive Classifier
classifier = PassiveAggressiveClassifier(C=1.0, random_state=42)  # You can experiment with the regularization parameter (C) and other parameters

# Train the classifier
classifier.fit(X_train_tfidf, y_train)

# Predictions
y_pred = classifier.predict(X_test_tfidf)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3395
           1       0.97      0.77      0.86       181

    accuracy                           0.99      3576
   macro avg       0.98      0.88      0.93      3576
weighted avg       0.99      0.99      0.99      3576



In [ ]:
import gensim.downloader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a classifier (e.g., Logistic Regression)
classifier = LogisticRegression()

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3395
           1       0.75      0.03      0.06       181

    accuracy                           0.95      3576
   macro avg       0.85      0.52      0.52      3576
weighted avg       0.94      0.95      0.93      3576



/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
import gensim.downloader
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Linear Support Vector Machine)
classifier = LinearSVC()

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3395
           1       0.71      0.03      0.05       181

    accuracy                           0.95      3576
   macro avg       0.83      0.51      0.51      3576
weighted avg       0.94      0.95      0.93      3576



In [ ]:
import gensim.downloader
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Random Forest)
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      3395
           1       0.99      0.40      0.57       181

    accuracy                           0.97      3576
   macro avg       0.98      0.70      0.78      3576
weighted avg       0.97      0.97      0.96      3576



In [ ]:
import gensim.downloader
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Gradient Boosting)
classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      3395
           1       0.91      0.33      0.48       181

    accuracy                           0.96      3576
   macro avg       0.94      0.66      0.73      3576
weighted avg       0.96      0.96      0.96      3576



In [ ]:
import gensim.downloader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., k-Nearest Neighbors)
classifier = KNeighborsClassifier(n_neighbors=5)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      3395
           1       0.83      0.50      0.62       181

    accuracy                           0.97      3576
   macro avg       0.90      0.75      0.80      3576
weighted avg       0.97      0.97      0.97      3576

